In [1]:
import pandas as pd

# ge15_2022.csv is a 2.7GB NATIONAL file (21.2M rows) - never load it whole.
# Pre-filtered to Negeri Sembilan via `grep ",Negeri Sembilan," ge15_2022.csv` into
# Data/ge15_2022_ns.csv (850,865 rows, verified during the original data review).
voter = "Data/ge15_2022_ns.csv"

df = pd.read_csv(voter)
print(df.shape)   # expect (850865, 11)
print(df.dtypes)
df.head(5)

(850865, 11)
uid           object
birth_year     int64
sex           object
ethnicity     object
state         object
parlimen      object
dun           object
dm_vr         object
dm            object
pm            object
saluran        int64
dtype: object


,uid,birth_year,sex,ethnicity,state,parlimen,dun,dm_vr,dm,pm,saluran
0,SUBHYBXYIL,1962,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,"Khemah A, IPD Jelebu (DUN Chennah)",1
1,AKSGJEKWFW,1963,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/03 Simpang Durian,126/01/00 Undi Awal,"Khemah A, IPD Jelebu (DUN Chennah)",1
2,VOKNZLPHGP,1963,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,"Khemah A, IPD Jelebu (DUN Chennah)",1
3,YYBWEDQIUK,1966,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/03 Simpang Durian,126/01/00 Undi Awal,"Khemah A, IPD Jelebu (DUN Chennah)",1
4,FXWUHPWPPG,1968,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,"Khemah A, IPD Jelebu (DUN Chennah)",1


In [2]:
results = "Data/ns_ge2022_results.csv"

df2 = pd.read_csv(results)
print(df2.shape)   # (864425, 11)
# print(df2.dtypes)
df2.head(5)

(1669, 17)


,PARLIMEN,DUN,NO. KOD DAERAH MENGUNDI,NAMA PUSAT MENGUNDI,SALURAN,KERTAS UNDI DALAM PETI UNDI (A),BN,PH,PN,PEJUANG,WARISAN,PSM,IND,IND.1,JUMLAH UNDI,UNDI YANG DITOLAK (C),KERTAS UNDI TIDAK DIMASUKKAN KE DALAM PETI UNDI (D)
0,P.126 JELEBU,NaN,UNDI POS,NaN,NaN,1245,599,168,299,5,0,0,0,0,1071,75,99
1,P.126 JELEBU,N01 CHENNAH,UNDI AWAL 126/01/00,"KHEMAH A, IPD JELEBU (DUN CHENNAH)",1.0,21,17,0,4,0,0,0,0,0,21,0,0
2,P.126 JELEBU,N02 PERTANG,UNDI AWAL 126/02/00,"KHEMAH B, IPD JELEBU (DUN PERTANG)",1.0,13,9,1,3,0,0,0,0,0,13,0,0
3,P.126 JELEBU,N03 SUNGAI LUI,UNDI AWAL 126/03/00,DEWAN PERDANA PULAPOL AIR HITAM,1.0,177,101,9,65,1,0,0,0,0,176,1,0
4,P.126 JELEBU,N03 SUNGAI LUI,UNDI AWAL 126/03/00,DEWAN PERDANA PULAPOL AIR HITAM,2.0,146,95,6,45,0,0,0,0,0,146,0,0


## Confirm inputs — check for the same classes of data-quality bugs found in Part A

Before trusting this file, re-run the checks that caught real bugs in
`ns_prn2023_results.csv`: drifting station codes (code changes per saluran instead of
staying fixed per station), multi-"meja" duplicate rows, and party-column swaps
(cross-checked against the paper's published seat totals).

In [3]:
results = df2.copy()
results['DUN'] = results['DUN'].str.strip()

# GE2022's postal rows are blank-DUN at the PARLIMEN level (not per-DUN like PRN2023) -
# already known from the original data review. Early rows contain 'UNDI AWAL'.
is_postal = results['NO. KOD DAERAH MENGUNDI'] == 'UNDI POS'
is_early = results['NO. KOD DAERAH MENGUNDI'].str.contains('UNDI AWAL', na=False)
results_ordinary = results[~(is_postal | is_early)].copy()

print(f"dropped {(is_postal | is_early).sum()} postal/early rows out of {len(results)}")
print(f"postal rows with blank DUN: {results[is_postal]['DUN'].isna().sum()} of {is_postal.sum()}")

# Extract dm_code - GE2022 uses unspaced "126/01/01" style (matches the roll's dm
# format directly, no " / " spacing to tolerate like PRN2023 needed).
code_parts = results_ordinary['NO. KOD DAERAH MENGUNDI'].str.extract(r'(\d+)\s*/\s*(\d+)\s*/\s*(\d+)')
results_ordinary['dm_code'] = code_parts[0] + '/' + code_parts[1] + '/' + code_parts[2]
assert results_ordinary['dm_code'].isna().sum() == 0, "some ordinary rows failed dm_code extraction"
results_ordinary['saluran'] = results_ordinary['SALURAN'].astype(int)

# Check 1: does the station code drift across a station's saluran rows?
drift_check = results_ordinary.groupby(['DUN', 'NAMA PUSAT MENGUNDI'])['dm_code'].nunique()
drifting = drift_check[drift_check > 1]
print(f"\nstations with a drifting code: {len(drifting)} of {len(drift_check)}")

# Check 2: multi-meja rows (same (DUN, dm_code, saluran), different vote counts)
dupe_check = results_ordinary.groupby(['DUN', 'dm_code', 'saluran']).size()
dupes = dupe_check[dupe_check > 1]
print(f"duplicate (DUN, dm_code, saluran) stream keys: {len(dupes)}")

# Check 3: no JUMLAH (seat-total) row
print(f"\nJUMLAH rows: {results['NO. KOD DAERAH MENGUNDI'].astype(str).str.contains('JUMLAH', case=False, na=False).sum()}")

dropped 76 postal/early rows out of 1669
postal rows with blank DUN: 8 of 8

stations with a drifting code: 0 of 379
duplicate (DUN, dm_code, saluran) stream keys: 199

JUMLAH rows: 0


In [4]:
# Check 4: party-column swap - cross-check aggregate PH/PN/BN seat totals against the
# paper's Table 3 "GE2022" columns for the 11 DAP seats (same technique that caught the
# N22 Rahang PN/PH swap in Part A).
paper_table3_ge2022 = {
    # dun: (DAP%, PN%, BN%)
    'N01 CHENNAH':          (43.3, 14.7, 40.4),
    'N08 BAHAU':             (66.1, 12.4, 19.8),
    'N10 NILAI':             (58.8, 24.6, 14.6),
    'N11 LOBAK':             (89.7, 2.5, 6.5),
    'N12 TEMIANG':           (65.3, 16.3, 16.3),
    'N21 BUKIT KEPAYANG':    (77.3, 11.1, 11.1),
    'N22 RAHANG':            (74.6, 9.9, 14.4),
    'N23 MAMBAU':            (83.4, 6.5, 9.6),
    'N24 SEREMBAN JAYA':     (74.5, 11.3, 13.3),
    'N30 LUKUT':             (74.0, 11.2, 13.0),
    'N36 REPAH':             (57.7, 15.3, 24.9),
}

for dun, (paper_dap, paper_pn, paper_bn) in paper_table3_ge2022.items():
    sub = results[results['DUN'] == dun]
    totals = sub[['PN', 'PH', 'BN', 'JUMLAH UNDI']].sum()
    our_pn = totals['PN'] / totals['JUMLAH UNDI'] * 100
    our_ph = totals['PH'] / totals['JUMLAH UNDI'] * 100
    our_bn = totals['BN'] / totals['JUMLAH UNDI'] * 100
    print(f"{dun:22s} ours PH={our_ph:5.1f} PN={our_pn:5.1f} BN={our_bn:5.1f}   "
          f"paper DAP={paper_dap:5.1f} PN={paper_pn:5.1f} BN={paper_bn:5.1f}")

N01 CHENNAH            ours PH= 43.8 PN= 14.9 BN= 40.9   paper DAP= 43.3 PN= 14.7 BN= 40.4
N08 BAHAU              ours PH= 66.5 PN= 12.6 BN= 20.2   paper DAP= 66.1 PN= 12.4 BN= 19.8
N10 NILAI              ours PH= 58.6 PN= 25.0 BN= 15.1   paper DAP= 58.8 PN= 24.6 BN= 14.6
N11 LOBAK              ours PH= 90.4 PN=  2.5 BN=  6.6   paper DAP= 89.7 PN=  2.5 BN=  6.5
N12 TEMIANG            ours PH= 59.8 PN= 19.4 BN= 19.4   paper DAP= 65.3 PN= 16.3 BN= 16.3
N21 BUKIT KEPAYANG     ours PH= 76.4 PN= 11.7 BN= 12.0   paper DAP= 77.3 PN= 11.1 BN= 11.1
N22 RAHANG             ours PH= 68.9 PN= 13.7 BN= 17.3   paper DAP= 74.6 PN=  9.9 BN= 14.4
N23 MAMBAU             ours PH= 83.9 PN=  6.3 BN=  9.7   paper DAP= 83.4 PN=  6.5 BN=  9.6
N24 SEREMBAN JAYA      ours PH= 75.1 PN= 11.4 BN= 13.4   paper DAP= 74.5 PN= 11.3 BN= 13.3
N30 LUKUT              ours PH= 74.6 PN= 11.3 BN= 13.2   paper DAP= 74.0 PN= 11.2 BN= 13.0
N36 REPAH              ours PH= 57.5 PN= 15.5 BN= 26.0   paper DAP= 57.7 PN= 15.3 BN= 24.9

## Ingest GE2022 results into long table

Collapse the 199 multi-"meja" duplicate rows (same fix as Part A's A4), then melt
`BN`/`PH`/`PN`/`PEJUANG`/`WARISAN`/`PSM`/`IND`/`IND.1` into long `party`/`votes` rows.

In [5]:
num_cols = ['BN', 'PH', 'PN', 'PEJUANG', 'WARISAN', 'PSM', 'IND', 'IND.1', 'JUMLAH UNDI',
            'KERTAS UNDI DALAM PETI UNDI (A)', 'UNDI YANG DITOLAK (C)']
results_collapsed = (
    results_ordinary
    .groupby(['DUN', 'dm_code', 'saluran'], as_index=False)[num_cols]
    .sum()
)

print(f"collapsed {len(results_ordinary)} rows -> {len(results_collapsed)} streams")

# Melt party columns into long form. Same "0 means didn't contest here" convention as
# Part A, so filter votes > 0 after melting. IND and IND.1 kept separate (Part A found
# some PRN2023 seats have two simultaneous independents; GE2022's IND.1 was confirmed
# always zero during the original data review, but keep both columns for consistency).
party_cols = ['BN', 'PH', 'PN', 'PEJUANG', 'WARISAN', 'PSM', 'IND', 'IND.1']
results_long = results_collapsed.melt(
    id_vars=['DUN', 'dm_code', 'saluran', 'JUMLAH UNDI'],
    value_vars=party_cols,
    var_name='party',
    value_name='votes',
)
results_long = results_long[results_long['votes'] > 0].copy()
results_long = results_long.rename(columns={'DUN': 'dun', 'JUMLAH UNDI': 'valid_votes'})
results_long['vote_share'] = results_long['votes'] / results_long['valid_votes']

print(results_long.shape)
results_long.head(10)

collapsed 1593 rows -> 1326 streams
(5470, 7)


,dun,dm_code,saluran,valid_votes,party,votes,vote_share
0,N01 CHENNAH,126/01/01,1,365,BN,202,0.553425
1,N01 CHENNAH,126/01/01,2,419,BN,191,0.455847
2,N01 CHENNAH,126/01/02,1,264,BN,115,0.435606
3,N01 CHENNAH,126/01/02,2,293,BN,180,0.614334
4,N01 CHENNAH,126/01/03,1,353,BN,227,0.643059
5,N01 CHENNAH,126/01/03,2,414,BN,167,0.403382
6,N01 CHENNAH,126/01/03,3,384,BN,147,0.382812
7,N01 CHENNAH,126/01/03,4,408,BN,167,0.409314
8,N01 CHENNAH,126/01/04,1,349,BN,223,0.638968
9,N01 CHENNAH,126/01/04,2,442,BN,338,0.764706


## Aggregate the GE2022 roll (NS-filtered) into ethnic composition per saluran

Same approach as A3: drop postal/early voters, extract a clean station code, bucket
ethnicity, compute each stream's ethnic composition + registered voter count.

In [6]:
roll = df.copy()

# Drop postal and early voters (see CLAUDE.md for why both are excluded)
is_postal_or_early = roll['dm'].str.contains('Undi Pos|Undi Awal', na=False)
roll_ordinary = roll[~is_postal_or_early].copy()

print(f"dropped {is_postal_or_early.sum()} postal/early voters "
      f"({is_postal_or_early.sum() / len(roll) * 100:.2f}% of {len(roll)})")

# Extract a clean station code from dm, e.g. "126/01/01 Kampong Sungai Buloh" -> "126/01/01"
roll_ordinary['dm_code'] = (
    roll_ordinary['dm']
    .str.extract(r'^(\d+\s*/\s*\d+\s*/\s*\d+)')[0]
    .str.replace(r'\s+', '', regex=True)
)
assert roll_ordinary['dm_code'].isna().sum() == 0, "some ordinary rows failed dm_code extraction"

# Bucket ethnicity into the paper's four categories (Orang Asli / Bumi Sabah / Bumi
# Sarawak / Other fold into "other")
ethnicity_map = {
    'Malay': 'malay',
    'Chinese': 'chinese',
    'Indian': 'indian',
    'Other': 'other',
    'Orang Asli': 'other',
    'Bumi Sabah': 'other',
    'Bumi Sarawak': 'other',
}
roll_ordinary['ethnic_group'] = roll_ordinary['ethnicity'].map(ethnicity_map)
assert roll_ordinary['ethnic_group'].isna().sum() == 0, "unmapped ethnicity value found"

dropped 20368 postal/early voters (2.39% of 850865)


In [7]:
# Group by (dun, dm_code, saluran) and count voters per ethnic group
stream_counts = (
    roll_ordinary
    .groupby(['dun', 'dm_code', 'saluran', 'ethnic_group'])
    .size()
    .unstack('ethnic_group', fill_value=0)
)

# Convert to percentages + keep registered count
stream_ethnic = stream_counts.copy()
stream_ethnic['n_registered'] = stream_counts.sum(axis=1)
for col in ['malay', 'chinese', 'indian', 'other']:
    stream_ethnic[f'pct_{col}'] = stream_counts[col] / stream_ethnic['n_registered']

stream_ethnic = stream_ethnic.reset_index()

# Sanity check: percentages should sum to ~1.0
pct_cols = [f'pct_{c}' for c in ['malay', 'chinese', 'indian', 'other']]
pct_sum = stream_ethnic[pct_cols].sum(axis=1)
bad = stream_ethnic[(pct_sum - 1).abs() > 0.02]
assert len(bad) == 0, f"{len(bad)} streams have ethnic percentages not summing to ~1.0"

print(stream_ethnic.shape)
stream_ethnic.head(5)

(1326, 12)


ethnic_group,dun,dm_code,saluran,chinese,indian,malay,other,n_registered,pct_malay,pct_chinese,pct_indian,pct_other
0,N01 CHENNAH,126/01/01,1,1,0,448,1,450,0.995556,0.002222,0.000000,0.002222
1,N01 CHENNAH,126/01/01,2,0,0,533,4,537,0.992551,0.000000,0.000000,0.007449
2,N01 CHENNAH,126/01/02,1,227,17,123,27,394,0.312183,0.576142,0.043147,0.068528
3,N01 CHENNAH,126/01/02,2,81,9,145,160,395,0.367089,0.205063,0.022785,0.405063
4,N01 CHENNAH,126/01/03,1,113,22,314,1,450,0.697778,0.251111,0.048889,0.002222


## B4: Merge results + ethnic composition

Same approach as A4: outer join on `(dun, dm_code, saluran)` with `indicator=True`,
assert zero unmatched rows.

In [8]:
merged = results_long.merge(
    stream_ethnic,
    on=['dun', 'dm_code', 'saluran'],
    how='outer',
    indicator=True,
)

print(merged['_merge'].value_counts())
unmatched = merged[merged['_merge'] != 'both']
assert len(unmatched) == 0, f"{len(unmatched)} rows failed to match between results and ethnic composition"
merged = merged.drop(columns='_merge')

print(merged.shape)
merged.head(10)

_merge
both          5470
left_only        0
right_only       0
Name: count, dtype: int64
(5470, 16)


,dun,dm_code,saluran,valid_votes,party,votes,vote_share,chinese,indian,malay,other,n_registered,pct_malay,pct_chinese,pct_indian,pct_other
0,N01 CHENNAH,126/01/01,1,365,BN,202,0.553425,1,0,448,1,450,0.995556,0.002222,0.000000,0.002222
1,N01 CHENNAH,126/01/01,1,365,PH,65,0.178082,1,0,448,1,450,0.995556,0.002222,0.000000,0.002222
2,N01 CHENNAH,126/01/01,1,365,PN,97,0.265753,1,0,448,1,450,0.995556,0.002222,0.000000,0.002222
3,N01 CHENNAH,126/01/01,1,365,PEJUANG,1,0.002740,1,0,448,1,450,0.995556,0.002222,0.000000,0.002222
4,N01 CHENNAH,126/01/01,2,419,BN,191,0.455847,0,0,533,4,537,0.992551,0.000000,0.000000,0.007449
5,N01 CHENNAH,126/01/01,2,419,PH,65,0.155131,0,0,533,4,537,0.992551,0.000000,0.000000,0.007449
6,N01 CHENNAH,126/01/01,2,419,PN,162,0.386635,0,0,533,4,537,0.992551,0.000000,0.000000,0.007449
7,N01 CHENNAH,126/01/01,2,419,PEJUANG,1,0.002387,0,0,533,4,537,0.992551,0.000000,0.000000,0.007449
8,N01 CHENNAH,126/01/02,1,264,BN,115,0.435606,227,17,123,27,394,0.312183,0.576142,0.043147,0.068528
9,N01 CHENNAH,126/01/02,1,264,PH,120,0.454545,227,17,123,27,394,0.312183,0.576142,0.043147,0.068528


## B5: Regression engine

Same logic as Part A's A5 — copied over rather than re-derived (the two notebooks don't
share state):
1. `seat_ethnic`: seat-level ethnic composition, weighted by each stream's
   `n_registered` — one row per DUN, used for the ≥20%-of-registered-voters reporting
   rule.
2. `estimate_support(group, pct_col)`: bivariate OLS via `np.polyfit` on a single
   `(dun, party)` group's streams, evaluated at `pct = 1.0`.
3. `cap_support`: clips to `[0, 1]`, Chinese capped at `0.99` to match the paper's
   Table 6 convention.
4. `run_regression_engine`: loops every `(dun, party)` pair in `merged`, zeroes out a
   group's estimate if `seat_ethnic` shows that group is under 20% of the seat's
   registered voters.

In [9]:
# Seat-level ethnic composition (weighted by each stream's registered voters), used
# for the paper's "only report a group if it's >=20% of registered voters" rule.
def weighted_pct(g, col):
    return (g[col] * g['n_registered']).sum() / g['n_registered'].sum()

seat_ethnic = stream_ethnic.groupby('dun').apply(
    lambda g: pd.Series({
        'malay': weighted_pct(g, 'pct_malay'),
        'chinese': weighted_pct(g, 'pct_chinese'),
        'indian': weighted_pct(g, 'pct_indian'),
        'other': weighted_pct(g, 'pct_other'),
        'n_registered': g['n_registered'].sum(),
    }),
    include_groups=False,
).reset_index()

print(seat_ethnic.shape)  # expect (36, 6)
seat_ethnic.head()

(36, 6)


,dun,malay,chinese,indian,other,n_registered
0,N01 CHENNAH,0.462745,0.443825,0.021259,0.072171,14535.0
1,N02 PERTANG,0.654829,0.201480,0.066433,0.077259,12840.0
2,N03 SUNGAI LUI,0.788987,0.062100,0.084770,0.064142,18615.0
3,N04 KLAWANG,0.648893,0.286950,0.038510,0.025647,12828.0
4,N05 SERTING,0.774610,0.127483,0.087125,0.010782,29957.0


In [10]:
import numpy as np

def estimate_support(group, ethnic_pct_col):
    """Bivariate OLS: vote_share ~ pct_<ethnic>, evaluated at x=1 (100%)."""
    if len(group) < 2 or group[ethnic_pct_col].nunique() < 2:
        return np.nan
    slope, intercept = np.polyfit(group[ethnic_pct_col], group['vote_share'], 1)
    return intercept + slope  # predicted vote_share at pct = 1.0


def cap_support(value, ethnic_group):
    if pd.isna(value):
        return value
    cap = 0.99 if ethnic_group == 'chinese' else 1.0
    return min(max(value, 0.0), cap)


def run_regression_engine(merged_df, seat_ethnic_df, threshold=0.20):
    """For every (dun, party), estimate Malay and Chinese support via the paper's
    bivariate OLS method, applying the cap and >=20%-of-registered-voters conventions."""
    records = []
    for (dun, party), group in merged_df.groupby(['dun', 'party']):
        seat_row = seat_ethnic_df.loc[seat_ethnic_df['dun'] == dun].iloc[0]
        row = {'dun': dun, 'party': party, 'n_streams': len(group)}
        for ethnic_group, pct_col in [('malay', 'pct_malay'), ('chinese', 'pct_chinese')]:
            estimate = cap_support(estimate_support(group, pct_col), ethnic_group)
            if seat_row[ethnic_group] < threshold:
                estimate = np.nan
            row[f'{ethnic_group}_support'] = estimate
        records.append(row)
    return pd.DataFrame(records)


support_table = run_regression_engine(merged, seat_ethnic)
print(support_table.shape)
support_table.head(20)

(159, 5)


,dun,party,n_streams,malay_support,chinese_support
0,N01 CHENNAH,BN,27,0.616510,0.141628
1,N01 CHENNAH,PEJUANG,21,0.005861,0.005386
2,N01 CHENNAH,PH,27,0.066502,0.851261
3,N01 CHENNAH,PN,27,0.311626,0.003866
4,N02 PERTANG,BN,26,0.616057,0.059826
5,N02 PERTANG,PEJUANG,21,0.006345,0.004824
6,N02 PERTANG,PH,26,0.091654,0.962620
7,N02 PERTANG,PN,26,0.286529,0.000000
8,N03 SUNGAI LUI,BN,35,0.596245,NaN
9,N03 SUNGAI LUI,PEJUANG,28,0.006279,NaN


## B6: Validate against the paper's published DAP GE2022 numbers

Table 6 has a GE2022 half we haven't checked yet ("Voting PH — GE2022" Malay/Chinese
columns for the same 11 `dap_seats.txt` seats). Same comparison approach as A6.

In [11]:
import re

dap_seats = pd.read_csv('dap_seats.txt', sep='\t', header=None,
                         names=['parlimen', 'dun', 'candidate', 'party'])
for col in dap_seats.columns:
    dap_seats[col] = dap_seats[col].str.strip()

# Normalize dun the same way script.py normalized the rolls: "N.01 Chennah" -> "N01 CHENNAH"
dap_seats['dun'] = dap_seats['dun'].apply(
    lambda v: re.sub(r'^([A-Za-z]+)\.(\d+)', r'\1\2', v).upper()
)

print(dap_seats.shape)  # expect (11, 4)
assert dap_seats['dun'].isin(seat_ethnic['dun']).all(), "some dap_seats DUNs didn't normalize to a known seat"
dap_seats[['dun', 'candidate']]

(11, 4)


,dun,candidate
0,N01 CHENNAH,LOKE SIEW FOOK
1,N08 BAHAU,TEO KOK SEONG
2,N10 NILAI,ARUL KUMAR A/L JAMBUNATHAN
3,N11 LOBAK,CHEW SEH YONG
4,N12 TEMIANG,NG CHIN TSAI
5,N21 BUKIT KEPAYANG,NICOLE TAN LEE KOON
6,N22 RAHANG,SIAU MEOW KONG
7,N23 MAMBAU,YAP YEW WENG
8,N24 SEREMBAN JAYA,GUNASEKAREN A/L PALASAMY
9,N30 LUKUT,CHOO KEN HWA


In [12]:
# Paper's Table 6, "Voting PH - GE2022" columns, NS rows only (transcribed from the PDF)
paper_table6_ns_ge2022 = pd.DataFrame([
    ('N01 CHENNAH',            6.8, 84.3),
    ('N08 BAHAU',             14.7, 99.0),
    ('N10 NILAI',             20.1, 99.0),
    ('N11 LOBAK',             np.nan, 94.5),
    ('N12 TEMIANG',           19.0, 94.6),
    ('N21 BUKIT KEPAYANG',    29.6, 99.0),
    ('N22 RAHANG',            26.9, 99.0),
    ('N23 MAMBAU',            np.nan, 99.0),
    ('N24 SEREMBAN JAYA',     12.4, 99.0),
    ('N30 LUKUT',             19.3, 98.7),
    ('N36 REPAH',              9.5, 99.0),
], columns=['dun', 'paper_malay', 'paper_chinese'])
paper_table6_ns_ge2022['paper_malay'] /= 100
paper_table6_ns_ge2022['paper_chinese'] /= 100

ours = support_table[(support_table['party'] == 'PH') & (support_table['dun'].isin(dap_seats['dun']))]

comparison = paper_table6_ns_ge2022.merge(ours, on='dun', how='left')
comparison['malay_diff_pp'] = (comparison['malay_support'] - comparison['paper_malay']) * 100
comparison['chinese_diff_pp'] = (comparison['chinese_support'] - comparison['paper_chinese']) * 100

assert len(comparison) == 11, f"expected 11 DAP seats, got {len(comparison)}"
comparison[['dun', 'paper_malay', 'malay_support', 'malay_diff_pp',
            'paper_chinese', 'chinese_support', 'chinese_diff_pp']]

,dun,paper_malay,malay_support,malay_diff_pp,paper_chinese,chinese_support,chinese_diff_pp
0,N01 CHENNAH,0.068,0.066502,-0.149796,0.843,0.851261,0.826124
1,N08 BAHAU,0.147,0.140988,-0.601170,0.990,0.990000,0.000000
2,N10 NILAI,0.201,0.115859,-8.514093,0.990,0.990000,0.000000
3,N11 LOBAK,NaN,NaN,NaN,0.945,0.953407,0.840734
4,N12 TEMIANG,0.190,0.136970,-5.302977,0.946,0.958837,1.283713
5,N21 BUKIT KEPAYANG,0.296,0.337488,4.148782,0.990,0.979926,-1.007412
6,N22 RAHANG,0.269,0.257968,-1.103206,0.990,0.990000,0.000000
7,N23 MAMBAU,NaN,NaN,NaN,0.990,0.988269,-0.173101
8,N24 SEREMBAN JAYA,0.124,0.105366,-1.863449,0.990,0.990000,0.000000
9,N30 LUKUT,0.193,0.165741,-2.725878,0.987,0.990000,0.300000


## B7: Apply the BN seat filter

**Important**: reuse the same 17 seat codes already identified from Part A's `merged`
— do NOT re-derive "which seats did BN contest" from this GE2022 data. GE2022 was a
*parliamentary* election; a `party == 'BN'` row here means BN contested that
*parliamentary* seat, not necessarily this specific DUN. The point of the Table 6-style
comparison is the same physical set of state seats across both elections.

In [13]:
# Same 17 seats identified in Part A's A7 (N9_State Election_analysis.ipynb), where
# BN fielded a state-seat candidate in PRN2023.
bn_seats = [
    'N02 PERTANG', 'N03 SUNGAI LUI', 'N05 SERTING', 'N06 PALONG', 'N07 JERAM PADANG',
    'N09 LENGGENG', 'N15 JUASSEH', 'N16 SERI MENANTI', 'N17 SENALING', 'N19 JOHOL',
    'N26 CHEMBONG', 'N27 RANTAU', 'N28 KOTA', 'N31 BAGAN PINANG', 'N32 LINGGI',
    'N34 GEMAS', 'N35 GEMENCHEH',
]
assert len(bn_seats) == 17
assert set(bn_seats).issubset(set(seat_ethnic['dun'])), "some Part A BN seat codes not found in this notebook's seats"

bn_support = support_table[(support_table['party'] == 'BN') & (support_table['dun'].isin(bn_seats))].sort_values('dun').reset_index(drop=True)
missing = set(bn_seats) - set(bn_support['dun'])
print(f"seats missing a GE2022 BN row: {missing}")

print(bn_support.shape)
bn_support

seats missing a GE2022 BN row: set()
(17, 5)


,dun,party,n_streams,malay_support,chinese_support
0,N02 PERTANG,BN,26,0.616057,0.059826
1,N03 SUNGAI LUI,BN,35,0.596245,NaN
2,N05 SERTING,BN,39,0.567433,NaN
3,N06 PALONG,BN,40,0.527287,NaN
4,N07 JERAM PADANG,BN,30,0.567916,NaN
5,N09 LENGGENG,BN,36,0.368451,NaN
6,N15 JUASSEH,BN,34,0.515289,NaN
7,N16 SERI MENANTI,BN,28,0.490088,NaN
8,N17 SENALING,BN,30,0.565042,NaN
9,N19 JOHOL,BN,31,0.541707,NaN


## B8: GE2022-side output table

Same shape as Part A's `bn_output`.

In [14]:
def format_pct(value):
    return "N.A." if pd.isna(value) else f"{value * 100:.1f}%"

bn_output_ge2022 = bn_support.copy()
seat_split = bn_output_ge2022['dun'].str.extract(r'^(N\d+)\s+(.*)$')
bn_output_ge2022.insert(0, 'seat_code', seat_split[0])
bn_output_ge2022.insert(1, 'seat_name', seat_split[1].str.title())
assert bn_output_ge2022[['seat_code', 'seat_name']].isna().sum().sum() == 0, "seat code/name split failed for some row"

bn_output_ge2022['malay_support_pct'] = bn_output_ge2022['malay_support'].apply(format_pct)
bn_output_ge2022['chinese_support_pct'] = bn_output_ge2022['chinese_support'].apply(format_pct)

bn_output_ge2022 = bn_output_ge2022[['seat_code', 'seat_name', 'n_streams', 'malay_support_pct', 'chinese_support_pct']]
bn_output_ge2022 = bn_output_ge2022.sort_values('seat_code').reset_index(drop=True)

print(bn_output_ge2022.shape)
bn_output_ge2022

(17, 5)


,seat_code,seat_name,n_streams,malay_support_pct,chinese_support_pct
0,N02,Pertang,26,61.6%,6.0%
1,N03,Sungai Lui,35,59.6%,N.A.
2,N05,Serting,39,56.7%,N.A.
3,N06,Palong,40,52.7%,N.A.
4,N07,Jeram Padang,30,56.8%,N.A.
5,N09,Lenggeng,36,36.8%,N.A.
6,N15,Juasseh,34,51.5%,N.A.
7,N16,Seri Menanti,28,49.0%,N.A.
8,N17,Senaling,30,56.5%,N.A.
9,N19,Johol,31,54.2%,N.A.
